# Notebook 05 — Profit Framework (Central Result)
### Per-customer expected-profit targeting (calibrated) vs global-threshold policies — realized profit on test

In [1]:

# ===== CELL 1: config =====
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
from sklearn.metrics import f1_score
 
CPROB_DIR = "/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-4"   # NB04 calibrated probs
SPLIT_DIR = "/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-1"   # NB01 splits
OUT_DIR   = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)
 
DATASETS   = ["maven", "cell"]
MODELS     = ["LogReg", "RF", "XGB", "LGBM"]
MONEY_COL  = {"maven": "Monthly Charge",   "cell": "MonthlyRevenue"}
TENURE_COL = {"maven": "Tenure in Months", "cell": "MonthsInService"}
 
# locked parameters (heterogeneous CLV)
MARGIN  = 0.30
TEN_MIN = 6        # floor on expected remaining months
TEN_MAX = 60       # cap on expected remaining months
DELTA   = 0.10     # offer cost as fraction of CLV
C_C     = 2.0      # FIXED contact cost
STRATEGY = "natural"

In [2]:

# ===== CELL 2 (optional): confirm NB04 path =====
for root,_,files in os.walk('/kaggle/input'):
    for f in files:
        if f.startswith('cprob_'): print(os.path.join(root,f)); break
 

/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-4/cprob_cell_LogReg_natural_isotonic_test.npy


In [3]:

# ===== CELL 3: empirical acceptance rate (gamma) from Cell2Cell =====
cell_train = pd.read_csv(f"{SPLIT_DIR}/cell_train.csv")
tot_calls  = cell_train["RetentionCalls"].sum()
tot_accept = cell_train["RetentionOffersAccepted"].sum()
GAMMA = float(np.clip(tot_accept / max(tot_calls, 1), 0.10, 0.90))
print(f"Empirical acceptance rate gamma = {tot_accept:.0f}/{tot_calls:.0f} = {GAMMA:.3f}")

Empirical acceptance rate gamma = 581/1168 = 0.497


In [4]:

# ===== CELL 4: profit primitives (heterogeneous CLV) =====
def clv(monthly, tenure, margin=MARGIN, ten_min=TEN_MIN, ten_max=TEN_MAX):
    m   = np.asarray(monthly, float); m   = np.where(np.isnan(m),   np.nanmedian(m),   m)
    tau = np.asarray(tenure,  float); tau = np.where(np.isnan(tau), np.nanmedian(tau), tau)
    L = np.clip(tau, ten_min, ten_max)          # expected remaining months ~ tenure
    return np.maximum(m, 0.0) * L * margin
 
def realized_profit(t, y, clv_v, gamma=None, delta=DELTA, c_c=C_C):
    g = GAMMA if gamma is None else gamma
    benefit = np.where(y == 1, g * clv_v * (1 - delta), -delta * clv_v)
    return float((t * (-c_c + benefit)).sum())
 
def ep_target(p, clv_v, gamma=None, delta=DELTA, c_c=C_C):
    g = GAMMA if gamma is None else gamma
    return -c_c + p * g * clv_v * (1 - delta) - (1 - p) * delta * clv_v
 
def best_profit_threshold(p_cal, y_cal, clv_cal, **kw):
    grid = np.linspace(0, 1, 201)
    prof = [realized_profit((p_cal >= t).astype(int), y_cal, clv_cal, **kw) for t in grid]
    return grid[int(np.argmax(prof))]
 
def best_f1_threshold(p_cal, y_cal):
    grid = np.linspace(0.01, 0.99, 99)
    f1 = [f1_score(y_cal, (p_cal >= t).astype(int)) for t in grid]
    return grid[int(np.argmax(f1))]
 
def oracle_profit(y, clv_v, gamma=None, delta=DELTA, c_c=C_C):
    g = GAMMA if gamma is None else gamma
    gain = g * clv_v * (1 - delta) - c_c
    t = ((y == 1) & (gain > 0)).astype(int)
    return realized_profit(t, y, clv_v, gamma=g, delta=delta, c_c=c_c)
 
 

In [5]:

# ===== CELL 5: load probs + money/tenure/labels =====
def load_combo(ds, model):
    base = f"{ds}_{model}_{STRATEGY}"
    d = {
        "p_cal_unc":  np.load(f"{CPROB_DIR}/cprob_{base}_none_cal.npy"),
        "p_test_unc": np.load(f"{CPROB_DIR}/cprob_{base}_none_test.npy"),
        "p_cal_cal":  np.load(f"{CPROB_DIR}/cprob_{base}_isotonic_cal.npy"),
        "p_test_cal": np.load(f"{CPROB_DIR}/cprob_{base}_isotonic_test.npy"),
    }
    cal_df  = pd.read_csv(f"{SPLIT_DIR}/{ds}_cal.csv")
    test_df = pd.read_csv(f"{SPLIT_DIR}/{ds}_test.csv")
    d["y_cal"]    = cal_df["target"].values
    d["y_test"]   = test_df["target"].values
    d["clv_cal"]  = clv(cal_df[MONEY_COL[ds]].values,  cal_df[TENURE_COL[ds]].values)
    d["clv_test"] = clv(test_df[MONEY_COL[ds]].values, test_df[TENURE_COL[ds]].values)
    return d
 

In [6]:

# ===== CELL 6: evaluate all policies =====
def eval_policies(ds, model):
    d = load_combo(ds, model)
    yT, clvT = d["y_test"], d["clv_test"]
    orc = oracle_profit(yT, clvT)
 
    def pack(name, t):
        prof = realized_profit(t, yT, clvT)
        return {"dataset": ds, "model": model, "policy": name,
                "profit": prof, "capture_%": 100 * prof / orc if orc > 0 else np.nan,
                "n_targeted": int(t.sum())}
 
    out = []
    out.append(pack("target_none", np.zeros_like(yT)))
    out.append(pack("target_all",  np.ones_like(yT)))
    out.append(pack("thr_0.5_unc", (d["p_test_unc"] >= 0.5).astype(int)))
    tf1 = best_f1_threshold(d["p_cal_unc"], d["y_cal"])
    out.append(pack("F1_thr_unc", (d["p_test_unc"] >= tf1).astype(int)))
    tp_u = best_profit_threshold(d["p_cal_unc"], d["y_cal"], d["clv_cal"])
    out.append(pack("profit_thr_unc", (d["p_test_unc"] >= tp_u).astype(int)))
    tp_c = best_profit_threshold(d["p_cal_cal"], d["y_cal"], d["clv_cal"])
    out.append(pack("profit_thr_cal", (d["p_test_cal"] >= tp_c).astype(int)))
    out.append(pack("percust_EP_unc", (ep_target(d["p_test_unc"], clvT) > 0).astype(int)))
    out.append(pack("percust_EP_cal", (ep_target(d["p_test_cal"], clvT) > 0).astype(int)))  # OURS
    return out, orc
 
rows, oracle_tbl = [], []
for ds in DATASETS:
    for model in MODELS:
        res, orc = eval_policies(ds, model)
        rows += res
        oracle_tbl.append({"dataset": ds, "model": model, "oracle_profit": orc})
 
results = pd.DataFrame(rows)
results["profit"]    = results["profit"].round(1)
results["capture_%"] = results["capture_%"].round(1)
results.to_csv(f"{OUT_DIR}/results_profit.csv", index=False)
 

In [7]:

# ===== CELL 7: headline — profit-capture % by policy (mean across models) =====
policy_order = ["target_none", "target_all", "thr_0.5_unc", "F1_thr_unc",
                "profit_thr_unc", "profit_thr_cal", "percust_EP_unc", "percust_EP_cal"]
pivot = (results.pivot_table(index="policy", columns="dataset",
                             values="capture_%", aggfunc="mean")
                 .reindex(policy_order).round(1))
print("\n===== PROFIT CAPTURE % (mean across models) — heterogeneous CLV =====")
print(pivot.to_string())
print("\nFull per-model table -> results_profit.csv")
print(results.to_string(index=False))
 
# quick deltas that matter for the paper
def cap(ds, pol): return pivot.loc[pol, ds]
for ds in DATASETS:
    print(f"\n[{ds}] percust_EP_cal vs percust_EP_unc (calibration gain): "
          f"{cap(ds,'percust_EP_cal') - cap(ds,'percust_EP_unc'):+.1f} pp")
    print(f"[{ds}] percust_EP_cal vs profit_thr_cal (per-customer gain): "
          f"{cap(ds,'percust_EP_cal') - cap(ds,'profit_thr_cal'):+.1f} pp")
    print(f"[{ds}] percust_EP_cal vs F1_thr_unc (profit-vs-accuracy): "
          f"{cap(ds,'percust_EP_cal') - cap(ds,'F1_thr_unc'):+.1f} pp")
 
 


===== PROFIT CAPTURE % (mean across models) — heterogeneous CLV =====
dataset         cell  maven
policy                     
target_none      0.0    0.0
target_all      36.1   10.3
thr_0.5_unc      7.9   45.4
F1_thr_unc      40.8   48.0
profit_thr_unc  41.3   58.4
profit_thr_cal  41.3   58.0
percust_EP_unc  41.2   60.0
percust_EP_cal  41.2   61.2

Full per-model table -> results_profit.csv
dataset  model         policy   profit  capture_%  n_targeted
  maven LogReg    target_none      0.0        0.0           0
  maven LogReg     target_all   8441.0       10.3        1318
  maven LogReg    thr_0.5_unc  35407.5       43.1         361
  maven LogReg     F1_thr_unc  32244.7       39.2         325
  maven LogReg profit_thr_unc  48088.3       58.5         626
  maven LogReg profit_thr_cal  48088.3       58.5         626
  maven LogReg percust_EP_unc  49492.6       60.2         569
  maven LogReg percust_EP_cal  49348.8       60.0         566
  maven     RF    target_none      0.0        0

In [8]:

# ===== CELL 8: sensitivity (champion model, our method) =====
CHAMP = {"maven": "LGBM", "cell": "XGB"}
 
def our_capture(ds, model, margin=MARGIN, ten_max=TEN_MAX, delta=DELTA, c_c=C_C, gamma=None):
    d = load_combo(ds, model)
    cal_df  = pd.read_csv(f"{SPLIT_DIR}/{ds}_cal.csv")   # unused but kept for clarity
    test_df = pd.read_csv(f"{SPLIT_DIR}/{ds}_test.csv")
    clvT = clv(test_df[MONEY_COL[ds]].values, test_df[TENURE_COL[ds]].values,
               margin=margin, ten_max=ten_max)
    yT = test_df["target"].values
    t = (ep_target(d["p_test_cal"], clvT, gamma, delta, c_c) > 0).astype(int)
    prof = realized_profit(t, yT, clvT, gamma, delta, c_c)
    orc  = oracle_profit(yT, clvT, gamma, delta, c_c)
    return 100 * prof / orc if orc > 0 else np.nan
 
sens = []
for ds in DATASETS:
    m = CHAMP[ds]
    for tm in [36, 48, 60]:
        sens.append(("ten_max", tm, ds, round(our_capture(ds, m, ten_max=tm), 1)))
    for mg in [0.20, 0.30, 0.40]:
        sens.append(("margin", mg, ds, round(our_capture(ds, m, margin=mg), 1)))
    for dl in [0.05, 0.10, 0.20]:
        sens.append(("offer_delta", dl, ds, round(our_capture(ds, m, delta=dl), 1)))
    for cc in [1.0, 2.0, 5.0]:
        sens.append(("contact_cc", cc, ds, round(our_capture(ds, m, c_c=cc), 1)))
    for gm in [max(GAMMA-0.1, 0.1), GAMMA, min(GAMMA+0.1, 0.9)]:
        sens.append(("gamma", round(gm, 2), ds, round(our_capture(ds, m, gamma=gm), 1)))
 
sens_df = pd.DataFrame(sens, columns=["param", "value", "dataset", "our_capture_%"])
sens_df.to_csv(f"{OUT_DIR}/results_profit_sensitivity.csv", index=False)
print("\n===== SENSITIVITY (our method, champion model, capture %) =====")
print(sens_df.to_string(index=False))
 
# ============================================================================
# DONE (v2). Compare CELL 7 deltas against v1: with heterogeneous CLV, the
# per-customer and calibration gains should be LARGER (if the mechanism is real).
# Report whatever the data shows — do not tune toward a target number.
# ============================================================================


===== SENSITIVITY (our method, champion model, capture %) =====
      param  value dataset  our_capture_%
    ten_max  36.00   maven           64.6
    ten_max  48.00   maven           62.9
    ten_max  60.00   maven           62.1
     margin   0.20   maven           62.5
     margin   0.30   maven           62.1
     margin   0.40   maven           62.2
offer_delta   0.05   maven           76.1
offer_delta   0.10   maven           62.1
offer_delta   0.20   maven           43.4
 contact_cc   1.00   maven           62.3
 contact_cc   2.00   maven           62.1
 contact_cc   5.00   maven           60.9
      gamma   0.40   maven           56.3
      gamma   0.50   maven           62.1
      gamma   0.60   maven           66.7
    ten_max  36.00    cell           42.4
    ten_max  48.00    cell           42.1
    ten_max  60.00    cell           42.1
     margin   0.20    cell           41.0
     margin   0.30    cell           42.1
     margin   0.40    cell           42.8
offer_delta

In [9]:
# ============================================================================
# CELL 9 (append to NB05 v2) — BOOTSTRAP CONFIDENCE INTERVALS
# Tests whether the small deltas (calibration gain, per-customer gain) are real
# or noise. Resamples the TEST set with replacement (B times); thresholds/rules
# stay fixed (already tuned on cal). Reports 95% CIs on capture% and on the key
# contrasts. NOTE: this captures TEST-SAMPLING uncertainty only, not split/seed
# uncertainty (that needs multi-seed).
# ============================================================================
B   = 2000
rng = np.random.RandomState(42)
POLICIES = ["F1_thr_unc", "profit_thr_cal", "percust_EP_unc", "percust_EP_cal"]

def get_decisions(ds, model):
    d = load_combo(ds, model)
    yT, clvT = d["y_test"], d["clv_test"]
    dec = {}
    tf1 = best_f1_threshold(d["p_cal_unc"], d["y_cal"])
    dec["F1_thr_unc"]     = (d["p_test_unc"] >= tf1).astype(int)
    tp_c = best_profit_threshold(d["p_cal_cal"], d["y_cal"], d["clv_cal"])
    dec["profit_thr_cal"] = (d["p_test_cal"] >= tp_c).astype(int)
    dec["percust_EP_unc"] = (ep_target(d["p_test_unc"], clvT) > 0).astype(int)
    dec["percust_EP_cal"] = (ep_target(d["p_test_cal"], clvT) > 0).astype(int)
    return dec, yT, clvT

# precompute decisions per dataset/model (fixed; bootstrap only re-weights rows)
store = {}
for ds in DATASETS:
    decs = {m: get_decisions(ds, m)[0] for m in MODELS}
    _, yT, clvT = get_decisions(ds, MODELS[0])
    store[ds] = (decs, yT, clvT)

def capture_for(decs, idx, yT, clvT, policy):
    orc = oracle_profit(yT[idx], clvT[idx])
    profs = [realized_profit(decs[m][policy][idx], yT[idx], clvT[idx]) for m in MODELS]
    return 100 * np.mean(profs) / orc if orc > 0 else np.nan

def ci(arr):
    a = np.asarray(arr)
    return np.nanmean(a), np.nanpercentile(a, 2.5), np.nanpercentile(a, 97.5)

ci_rows = []
for ds in DATASETS:
    decs, yT, clvT = store[ds]
    n = len(yT)
    boot  = {p: [] for p in POLICIES}
    dboot = {"cal_gain": [], "percust_gain": [], "profit_vs_acc": []}
    for _ in range(B):
        idx = rng.randint(0, n, n)
        caps = {p: capture_for(decs, idx, yT, clvT, p) for p in POLICIES}
        for p in POLICIES:
            boot[p].append(caps[p])
        dboot["cal_gain"].append(caps["percust_EP_cal"] - caps["percust_EP_unc"])
        dboot["percust_gain"].append(caps["percust_EP_cal"] - caps["profit_thr_cal"])
        dboot["profit_vs_acc"].append(caps["percust_EP_cal"] - caps["F1_thr_unc"])

    for p in POLICIES:
        m, lo, hi = ci(boot[p])
        ci_rows.append((ds, "capture%", p, round(m, 1), round(lo, 1), round(hi, 1)))
    for k, v in dboot.items():
        m, lo, hi = ci(v)
        sig = "yes" if (lo > 0 or hi < 0) else "NO"
        ci_rows.append((ds, "delta_pp", k, round(m, 2), round(lo, 2), round(hi, 2)))

ci_df = pd.DataFrame(ci_rows, columns=["dataset", "kind", "metric", "mean", "ci_lo", "ci_hi"])
ci_df["significant"] = ci_df.apply(
    lambda r: ("—" if r["kind"] == "capture%"
               else ("yes" if (r["ci_lo"] > 0 or r["ci_hi"] < 0) else "NO")), axis=1)
ci_df.to_csv(f"{OUT_DIR}/results_profit_ci.csv", index=False)
print(f"===== BOOTSTRAP 95% CIs (B={B}, mean across models) =====")
print(ci_df.to_string(index=False))
print("\nInterpret: for a delta, if [ci_lo, ci_hi] excludes 0 (significant=yes),")
print("the effect is real at ~95%; if it spans 0 (NO), report as not significant.")

===== BOOTSTRAP 95% CIs (B=2000, mean across models) =====
dataset     kind         metric  mean  ci_lo  ci_hi significant
  maven capture%     F1_thr_unc 48.00  41.20  54.60           —
  maven capture% profit_thr_cal 57.90  51.70  64.00           —
  maven capture% percust_EP_unc 59.90  53.40  66.30           —
  maven capture% percust_EP_cal 61.00  54.30  67.30           —
  maven delta_pp       cal_gain  1.12  -1.21   3.37          NO
  maven delta_pp   percust_gain  3.12  -0.14   6.47          NO
  maven delta_pp  profit_vs_acc 13.09   6.71  19.09         yes
   cell capture%     F1_thr_unc 40.70  38.10  43.30           —
   cell capture% profit_thr_cal 41.30  38.50  44.20           —
   cell capture% percust_EP_unc 41.20  38.40  44.00           —
   cell capture% percust_EP_cal 41.20  38.30  44.10           —
   cell delta_pp       cal_gain  0.05  -0.61   0.73          NO
   cell delta_pp   percust_gain -0.06  -0.63   0.56          NO
   cell delta_pp  profit_vs_acc  0.47  -1.05 